In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np
from itertools import combinations
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *
import re
load_dotenv()

pd.set_option('display.max_rows', None)

model_ = "gpt-5.1"
log_name_ = "credit_seed1_03_homonymous"
chunk_size_ = 1

llm = llm_call(model_version = model_, api_key= os.getenv("API_KEY"))
df_new, cases_json = build_event_jsons(log_name = f"./dataset/{log_name_}.csv", chunk_cases = chunk_size_)

## 🔍 Step 1: Process Flow Context Extraction & Candidate Filtering

This step leverages the `pm4py` library to analyze the process behavior (Heuristics Net) and identify potential **Homonym** activity candidates.

### Key Functional Components
1. **Heuristics Net Discovery**: Analyzes the Direct Follows Graph (DFG) to establish predecessor/successor relationships and their respective frequencies.
2. **Contextual Structuring**: Constructs a structured profile for each activity, including **Predecessors** and **Successors** sorted by occurrence frequency.
3. **Homonym Candidate Filtering**:
    - **Merge Point Criterion**: Selects activities with 2 or more distinct predecessors (indicating potential context merging).
    - **Re-occurrence Criterion**: Filters for activities that appear 2 or more times within a single `case_id`.

### Expected Outputs
- `flow_all`: A comprehensive list containing the flow context for every activity in the log.
- `flow_filtered`: A refined list of activities that satisfy the specific structural criteria for potential homonyms.

In [2]:
def homonym_step1(df: pd.DataFrame,
                  case_col: str = 'case_id',
                  time_col: str = 'timestamp',
                  act_col: str = 'activity',
                  filter_list: set = None):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    heu_net = pm4py.discover_heuristics_net(df_pm4py)
    temp_preds = defaultdict(list)
    temp_succs = defaultdict(list)
    for (src, dst), freq in heu_net.dfg.items():
        temp_succs[src].append((dst, freq))
        temp_preds[dst].append((src, freq))
    flow_data_list = []
    flow_data_list_filtered = [] 
    all_activities = sorted(df[act_col].unique())
    for act in all_activities:
        if filter_list is not None and act not in filter_list:
            continue
        pred_list = [item[0] for item in sorted(temp_preds[act], key=lambda x: x[1], reverse=True)]
        succ_list = [item[0] for item in sorted(temp_succs[act], key=lambda x: x[1], reverse=True)]
        
        context_item = {
            'activity': act,
            'predecessors': pred_list,
            'successors': succ_list
        }
        flow_data_list.append(context_item)
        if len(pred_list) >= 2:
            flow_data_list_filtered.append(context_item)
    counts = df.groupby([case_col, act_col]).size().reset_index(name='count')
    activities_appearing_twice = counts[counts['count'] >= 2][act_col].unique()
    flow_data_list_final = [
        item for item in flow_data_list_filtered 
        if item['activity'] in activities_appearing_twice
    ]
    return flow_data_list, flow_data_list_final

flow_all, flow_filtered = homonym_step1(df_new)


## 🧩 Step 2: Combinatorial Candidate Search & Structural Alignment

This step identifies the best sets of existing activities that can "reconstruct" the flow context of a target homonym. It assumes that a homonym is a label representing a mixture of other functional steps.

### Algorithm Overview
The goal is to find a combination of $r$ activities ($2 \le r \le 5$) whose combined predecessors and successors most closely mirror the target's environment.

1. **Redundancy Filtering**: Before evaluating a combination, the script checks if each member contributes unique, valid information. If a member only introduces "noise" (flows the target doesn't have) without adding "coverage" (flows the target does have), the combination is discarded.
2. **Structural Error Calculation**:
    - **Symmetric Difference ($\Delta$)**: We calculate the error count using the set symmetric difference between the Target's context and the Union of the Combo's context ($Target \oplus Combined$).
    - **Logic**: A lower error count indicates that the combination explains the target's behavior with minimal missing or extra flows.
3. **Selection Criteria**:
    - **Minimum Threshold**: A valid combination must cover at least 2 of the target's predecessors and 1 successor.
    - **Top-N Ranking**: Candidates are ranked by error count. The script selects the top 5 combinations (including ties at the 5th position) for further semantic validation.

### Expected Outputs
- `homonym_candidates`: A dictionary mapping each target activity to a list of potential activity clusters (combinations) that could replace it.

In [3]:
def homonym_step2(target_activity, flow_others):
    target_pre = set(target_activity['predecessors'])
    target_suc = set(target_activity['successors'])
    target_name = target_activity['activity']
    all_results = [] 
    for r in range(2, 6):
        for combo in combinations(flow_others, r):
            is_redundant_combo = False
            for i, act_item in enumerate(combo):
                others = [x for j, x in enumerate(combo) if i != j]
                union_pre_others = set().union(*[set(x['predecessors']) for x in others])
                union_suc_others = set().union(*[set(x['successors']) for x in others])
                my_pre = set(act_item['predecessors'])
                my_suc = set(act_item['successors'])
                new_correct_pre = (my_pre & target_pre) - union_pre_others
                new_correct_suc = (my_suc & target_suc) - union_suc_others
                my_extra_pre = my_pre - target_pre
                my_extra_suc = my_suc - target_suc
                if (not new_correct_pre and not new_correct_suc) and (my_extra_pre or my_extra_suc):
                    is_redundant_combo = True
                    break
            if is_redundant_combo:
                continue
            combined_pre = set().union(*[set(x['predecessors']) for x in combo])
            combined_suc = set().union(*[set(x['successors']) for x in combo])
            pre_intersection = target_pre & combined_pre
            suc_intersection = target_suc & combined_suc
            if len(pre_intersection) < 2 or len(suc_intersection) < 1:
                continue
            pre_diff = target_pre ^ combined_pre
            suc_diff = target_suc ^ combined_suc
            total_error = len(pre_diff) + len(suc_diff)
            all_results.append({
                "target": target_name,
                "matched_activities": [x['activity'] for x in combo],
                "error_count": total_error
            })
    if not all_results:
        return []
    sorted_results = sorted(all_results, key=lambda x: x['error_count'])
    if len(sorted_results) <= 5:
        return sorted_results
    fifth_error_val = sorted_results[4]['error_count']
    filtered_results = [res for res in sorted_results if res['error_count'] <= fifth_error_val]
    return filtered_results
    
homonym_candidates = {}
for target in flow_filtered:
    flow_others_ = [item for item in flow_all if item['activity'] != target['activity']]
    matches = homonym_step2(target, flow_others_)
    if matches:
        target_name = target['activity']
        homonym_candidates[target_name] = [m['matched_activities'] for m in matches]


## 🧠 Step 3: Semantic Validation via LLM (Semantic Fusion)

While Step 2 identifies structural candidates, this step employs a Large Language Model (LLM) to perform **Semantic Reconstruction**. It ensures that the candidate groups are not just structurally similar but also logically represent the "True Identity" of the homonymous activity.

### The "Semantic Fusion" Approach
The model is instructed to synthesize two critical dimensions:
1. **Lexical Meaning**: The general business definition of the activity name.
2. **Contextual Meaning**: The functional role revealed by its specific predecessors (triggers) and successors (outputs).

### Validation Logic
The LLM evaluates the candidates based on:
- **Reconstruction Match**: Whether the combined 'True Semantic Identities' of all group members collectively cover the target activity's purpose.
- **Ambiguity Resolution**: Determining if the target name serves as a vague "umbrella term" for the specific functional roles performed by the candidate members.

### Process Flow
- Data for the target and each candidate combo is serialized into JSON format.
- The LLM performs an "All-or-Nothing" validation based on the synthesis of names and process flows.
- Validated matches are stored in `final_validated_homonyms` for final structural arbitration.

In [4]:
SYSTEM_PROMPT_HOMONYM_STEP3 = """
You are a Process Mining Expert specializing in Semantic Reconstruction.
Your goal is to validate if a group of activities is a 'Homonymous Decomposition' of a Target Activity by synthesizing their literal names and process contexts.

### CORE LOGIC: THE SEMANTIC FUSION
Do not treat names and contexts separately. You must RECONSTRUCT the 'True Identity' of each activity as follows:
1. LEXICAL MEANING: What does the activity name (e.g., 'Review') imply in a general business sense?
2. CONTEXTUAL MEANING: What do the Predecessors (triggers) and Successors (outputs) reveal about its specific role in this process?
3. SYNTHESIS (THE TRUE SEMANTIC): Combine 1 & 2 to define the "Real-World Action" being performed.

### VALIDATION CRITERIA
- RECONSTRUCTION MATCH: Does the 'True Semantic Identity' of the Target Activity encompass the 'Combined True Semantic Identities' of ALL members in the candidate group?
- AMBIGUITY RESOLUTION: Is the Target name a vague "umbrella term" that effectively describes the specific functional roles revealed by the members' contexts?

### STRICT CONSTRAINTS
- ALL-OR-NOTHING: Evaluate the group as a whole. Do not modify the list.
- NO PROSE: Output ONLY the JSON object.
"""

def get_homonym_user_prompt_step3(target_activity_json, flow_others_json):
    target_data = json.loads(target_activity_json)
    flow_others_data = json.loads(flow_others_json)
    target_name = target_data['target']['homonymous_activity']
    input_members = [m['member_activity'] for m in flow_others_data['flow_others']]
    return f"""
### TASK: Comprehensive Semantic Validation

**OBJECTIVE:**
Analyze if the set {input_members} is the specific realization of the homonymous activity "{target_name}".

**INPUT DATA:**
1. [TARGET ACTIVITY]: {target_activity_json}
2. [FLOW OTHERS] (Candidates to validate): {flow_others_json}

**EXECUTION STEPS:**
1. **Target Identity Reconstruction:** Combine the name "{target_name}" with its Pre/Suc. Define exactly what "True Action" this activity represents here.
2. **Member Identity Reconstruction:** For each activity in {input_members}, combine its name with its Pre/Suc. Define the "True Action" of each member.
3. **Synthesis & Comparison:** - Does the collective "True Action" of these members explain why they might have been incorrectly grouped under the name "{target_name}"?
   - Is there a functional alignment between the Target's context and the Members' combined context?

**STRICT OUTPUT FORMAT:**
- If Match: {{ "found": true, "data": {{ "homonymous_label": "{target_name}", "member_activities": {input_members} }} }}
- If No Match: {{ "found": false, "data": [] }}
"""
    
flow_lookup = {item['activity']: item for item in flow_all}
final_validated_homonyms = {}
for target_name, candidates in homonym_candidates.items():
    target_info = flow_lookup.get(target_name)
    target_data = {
        "homonymous_activity": target_name,
        "predecessors": target_info['predecessors'],
        "successors": target_info['successors']
    }
    for i, combo in enumerate(candidates, 1):
        combo_details = []
        for member_name in combo:
            member_info = flow_lookup.get(member_name)
            combo_details.append({
                "member_activity": member_name,
                "predecessors": member_info['predecessors'],
                "successors": member_info['successors']
            })
        target_json = json.dumps({"target": target_data}, indent=2, ensure_ascii=False)
        flow_others_json = json.dumps({"flow_others": combo_details}, indent=2, ensure_ascii=False)
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT_HOMONYM_STEP3},
            {"role": "user", "content": get_homonym_user_prompt_step3(target_json, flow_others_json)}
        ]
        raw_output = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
        try:
            res = json.loads(raw_output) if isinstance(raw_output, str) else raw_output
            if res.get('found'):
                label = res['data']['homonymous_label']
                members = res['data']['member_activities']
                if label not in final_validated_homonyms:
                    final_validated_homonyms[label] = []
                final_validated_homonyms[label].append(members)
        except Exception as e:
            print(f"⚠️ Error parsing LLM output: {e}")

## ⚖️ Step 4: Final Structural Arbitration & Selection

In this final phase, we perform a **Structural Audit** to select the single best candidate combination from the semantically validated list. 

### The Heuristics Net Summary
To ensure the reconstruction is grounded in actual process behavior, we generate a **Heuristics Net Summary**. This acts as the map, providing the LLM with a global view of every activity's frequency and routing in the original event log.

### Decision Logic: The Comparative Audit
The LLM acts as an arbitrator, evaluating multiple candidate combos based on the following structural criteria:
1. **Structural Fit**:
    - **Maximize Coverage**: How many of the Target's original predecessors and successors are correctly represented by the combo?
    - **Minimize Noise**: Does the combo introduce "ghost flows" (extra arrows) that didn't exist for the Target?
2. **Phase Alignment**: Ensures the suggested members logically operate in the same stage of the business process as the Target.
3. **Behavioral Equivalence**: Verifies that replacing the Target with this combo maintains a clean, straightforward process model without creating "spaghetti" flows.

### Output
- `homonym_predict`: The final mapping dictionary where each **Target Activity (Key)** is mapped to its **Optimal Member List (Value)**.

In [5]:
def get_heuristics_summary(df: pd.DataFrame,
                            case_col: str = 'case_id',
                            time_col: str = 'timestamp',
                            act_col: str = 'activity'):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"])
    heu_net = pm4py.discover_heuristics_net(df_pm4py)
    
    return "\n".join(f"{src} -> {dst} (Freq: {freq})" for (src, dst), freq in heu_net.dfg.items())
heu_sum = get_heuristics_summary(df_new)

In [17]:
SYSTEM_PROMPT_HOMONYM_STEP4 = """
You are a Process Mining Expert specializing in Structural Coherence.
Your goal is to select the SINGLE BEST candidate combo for a Target Activity by performing a comparative audit.

### ANALYSIS PROTOCOL
Before selecting the best combo, you must mentally:
1. **Audit Each Combo**: Compare the collective Predecessors/Successors of the combo members against the [HEURISTICS NET SUMMARY].
2. **Evaluate Structural Fit**: Identify which combo minimizes "Noise" (extra arrows) and maximizes "Coverage" (matching arrows).
3. **Check Phase Consistency**: Ensure the members operate in the same business phase as the Target.

### SELECTION CRITERIA
- **Phase Alignment**: Do the candidate members naturally operate in the exact same process phase as the Target?
- **Structural Fit**: Which combo minimizes "Noise" (extra predecessors/successors the Target didn't have) and maximizes "Coverage" (matching the Target's actual flow)?
- **Structural Simplicity**: If the Target is replaced by this combo, does the overall process flow remain clear and straightforward?
- **Behavioral Equivalence**: Does the combo collectively fulfill the exact routing purpose of the Target without introducing illogical detours?

### STRICT CONSTRAINTS
- **NO MIX AND MATCH**: Select exactly ONE existing combo from the provided list.
- **EXACT MATCH**: The "original_activity" must be an exact copy of the chosen combo's members.
- **NO REASONING**: Do NOT include any reasoning, analysis, or extra fields in the output.

### OUTPUT FORMAT
Output ONLY a JSON object. No markdown, no prose.
{
  "homonymous_label": "Target Name",
  "original_activity": ["Member A", "Member B"]
}
"""

def get_homonym_user_prompt_step4(heu_sum, target_data, all_candidates_data):
    return f"""
### [HEURISTICS NET SUMMARY: THE GROUND TRUTH]
{heu_sum}

### [TARGET ACTIVITY: THE LABEL TO RESOLVE]
- **Name**: "{target_data['activity']}"
- **Predecessors**: {target_data['pre']}
- **Successors**: {target_data['suc']}

### [ALL CANDIDATE COMBOS TO EVALUATE]
{json.dumps(all_candidates_data, indent=2, ensure_ascii=False)}

### [YOUR TASK]
1. Perform a comparative audit of the combos above.
2. Select the single best combo that logically replaces "{target_data['activity']}" with the cleanest process flow.
3. Return the final selection in the requested JSON format. NO PROSE, NO REASONING.
"""

flow_lookup = {item['activity']: item for item in flow_all}
final_best_matches = {}

homonym_predict = {}
for target_name, candidates in final_validated_homonyms.items():
    target_info = {
        "activity": target_name,
        "pre": flow_lookup[target_name]['predecessors'],
        "suc": flow_lookup[target_name]['successors']
    }
        
    all_combos_details = [
        {
            "combo_id": i,
            "combo_members": combo,
            "member_details": [
                {
                    "member_name": member,
                    "predecessors": flow_lookup.get(member, {}).get('predecessors', []),
                    "successors": flow_lookup.get(member, {}).get('successors', [])
                } for member in combo
            ]
        } for i, combo in enumerate(candidates, 1)
    ]
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT_HOMONYM_STEP4},
        {"role": "user", "content": get_homonym_user_prompt_step4(heu_sum, target_info, all_combos_details)}
    ]
    response = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)        
    homonym_predict[response['homonymous_label']]=response['original_activity']

In [18]:
df_homonym = df_new[df_new['label'].notna()].copy()
df_homonym['clean_activity'] = df_homonym['label'].str.extract(r'\((.*?)\)')
homonym_answer = (
    df_homonym.groupby('activity')['clean_activity']
    .unique()
    .apply(list)
    .to_dict()
)

print("------------------PREDICTION------------------")
print(json.dumps(homonym_predict, indent=4, ensure_ascii=False))
print("------------------ANSWER------------------")
print(json.dumps(homonym_answer, indent=4, ensure_ascii=False))


------------------PREDICTION------------------
{
    "Decision review": [
        "Make decision",
        "review request received"
    ],
    "Information exchange": [
        "Request info",
        "info received"
    ],
    "Verification process": [
        "Perform checks",
        "Request info",
        "info received"
    ]
}
------------------ANSWER------------------
{
    "Decision review": [
        "Make decision",
        "review request received"
    ],
    "Information exchange": [
        "info received",
        "Request info"
    ],
    "Verification process": [
        "Perform checks",
        "Check for completeness"
    ]
}


In [19]:
def evaluate_homonym_results(answer_dict, predict_dict):
    ans_keys = set(answer_dict.keys())
    pred_keys = set(predict_dict.keys())
    
    tp_keys = ans_keys.intersection(pred_keys)
    fp_keys = pred_keys - ans_keys
    fn_keys = ans_keys - pred_keys
    
    key_precision = len(tp_keys) / len(pred_keys) if pred_keys else 0
    key_recall = len(tp_keys) / len(ans_keys) if ans_keys else 0
    key_f1 = (2 * key_precision * key_recall) / (key_precision + key_recall) if (key_precision + key_recall) else 0
    all_v_f1 = []
    all_v_precision = []
    all_v_recall = []

    for key in tp_keys:
        ans_vals = set(answer_dict[key])
        pred_vals = set(predict_dict[key])
        
        tp_v = ans_vals.intersection(pred_vals)
        
        v_prec = len(tp_v) / len(pred_vals) if pred_vals else 0
        v_reca = len(tp_v) / len(ans_vals) if ans_vals else 0
        v_f1 = (2 * v_prec * v_reca) / (v_prec + v_reca) if (v_prec + v_reca) else 0
        
        all_v_precision.append(v_prec)
        all_v_recall.append(v_reca)
        all_v_f1.append(v_f1)
        
    avg_v_precision = sum(all_v_precision) / len(tp_keys) if tp_keys else 0
    avg_v_recall = sum(all_v_recall) / len(tp_keys) if tp_keys else 0
    avg_v_f1 = sum(all_v_f1) / len(tp_keys) if tp_keys else 0

    # 결과 리포트 출력
    print("-" * 50)
    print("      [Homonymous Pattern Detection Evaluation Report]")
    print("-" * 50)
    print(f"1. Key Selection (Clean Activity Identification)")
    print(f"   - Precision : {key_precision:.4f}")
    print(f"   - Recall    : {key_recall:.4f}")
    print(f"   - F1-Score  : {key_f1:.4f}")
    print("-" * 50)
    print(f"2. Value Selection (Homonyms Variation Mapping Accuracy - Average)")
    print(f"   - Avg Precision : {avg_v_precision:.4f}")
    print(f"   - Avg Recall    : {avg_v_recall:.4f}")
    print(f"   - Avg F1-Score  : {avg_v_f1:.4f}")
    print("-" * 50)
    print(f"   * Analyzed Keys: {len(tp_keys)} matched / {len(ans_keys)} total")
    print("-" * 50)

    return {
        "key_metrics": {"precision": key_precision, "recall": key_recall, "f1": key_f1},
        "value_metrics": {"avg_precision": avg_v_precision, "avg_recall": avg_v_recall, "avg_f1": avg_v_f1}
    }
    
print(evaluate_homonym_results(homonym_answer, homonym_predict))

--------------------------------------------------
      [Homonymous Pattern Detection Evaluation Report]
--------------------------------------------------
1. Key Selection (Clean Activity Identification)
   - Precision : 1.0000
   - Recall    : 1.0000
   - F1-Score  : 1.0000
--------------------------------------------------
2. Value Selection (Homonyms Variation Mapping Accuracy - Average)
   - Avg Precision : 0.7778
   - Avg Recall    : 0.8333
   - Avg F1-Score  : 0.8000
--------------------------------------------------
   * Analyzed Keys: 3 matched / 3 total
--------------------------------------------------
{'key_metrics': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}, 'value_metrics': {'avg_precision': 0.7777777777777777, 'avg_recall': 0.8333333333333334, 'avg_f1': 0.7999999999999999}}
